<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drexelwireless/ece512-courseware/blob/main/notebooks/lectures/week05-small-scale-fading.ipynb)

# Week 5 — Small-Scale Fading
**ECE 512 Wireless Communications** · companion notebook to the Week 5 lecture

This notebook lets you experiment with the ideas from the slides:

1. Doppler shift $f_{D,n} = f_m\cos\theta_n$
2. Fading signal model: flat vs. frequency-selective fading
3. I/Q decomposition and the central limit theorem
4. Rayleigh fading: envelope and power distributions
5. Fade margin and outage
6. Ricean fading and the $K$-factor
7. Doppler power spectrum and autocorrelation
8. Fading simulators: sum of sinusoids vs. Clarke/Gans

Run the setup cell first. In Colab: *Runtime → Run all*.

In [ ]:
# --- Setup: installs the course package in Google Colab (safe to re-run) ---
COURSEWARE_URL = "git+https://github.com/drexelwireless/ece512-courseware"
import importlib.util
if importlib.util.find_spec("ece512") is None:
    get_ipython().run_line_magic("pip", f"install -q {COURSEWARE_URL}")

import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import welch
from ipywidgets import interact, IntSlider, FloatSlider
from ece512 import fading, smallscale as ss

def db20(x):
    """Envelope in dB (20 log10), floored to avoid log(0)."""
    return 20 * np.log10(np.maximum(np.abs(x), 1e-6))

## 1. Doppler shift
A mobile moving at speed $v$ travels $\Delta l = v\,\Delta t\cos\theta$ further toward (or away from) a plane wave
arriving at angle $\theta$ to the direction of motion. The phase changes by $\Delta\phi = \frac{2\pi}{\lambda}\Delta l$, so
the received frequency is shifted by

$$f_{D,n} = \frac{1}{2\pi}\frac{\Delta\phi}{\Delta t} = f_m\cos\theta_n, \qquad f_m = \frac{v}{\lambda} = f_c\,\frac{v}{c}.$$

Waves from ahead ($\theta = 0$) are shifted up by $f_m$, waves from behind ($\theta = 180^\circ$) down by $f_m$, and waves
from the side are not shifted. First, the two examples from the slides:

In [ ]:
examples = [("5.2 GHz WLAN, pedestrian", 5.2e9, 5), ("GSM 900, highway", 900e6, 110)]
for name, fc, v_kmh in examples:
    print(f"{name:26s} v = {v_kmh:3d} km/h ({v_kmh / 3.6:4.1f} m/s):  f_m = {ss.max_doppler_hz(v_kmh / 3.6, fc):5.1f} Hz")

Move the sliders: the left panel shows the mobile (moving along $+x$) and a plane wave arriving from angle $\theta$;
the right panel shows where that wave lands on the $f_D = f_m\cos\theta$ curve.

In [ ]:
def show_doppler(speed_kmh=60.0, fc_ghz=2.0, theta_deg=45.0):
    fm = ss.max_doppler_hz(speed_kmh / 3.6, fc_ghz * 1e9)
    th = np.deg2rad(theta_deg)
    fd = ss.doppler_shift_hz(fm, th)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.6), gridspec_kw={"width_ratios": [1, 1.4]})
    # geometry: wavefronts perpendicular to the arrival direction u
    u = np.array([np.cos(th), np.sin(th)]); w = np.array([-u[1], u[0]])
    for s in np.arange(0.4, 2.1, 0.35):
        p = s * u
        ax1.plot(*np.c_[p - 0.45 * w, p + 0.45 * w], color="tab:blue", alpha=0.5, lw=1.5)
    ax1.annotate("", xy=(0.15 * u[0], 0.15 * u[1]), xytext=(2.1 * u[0], 2.1 * u[1]),
                 arrowprops=dict(arrowstyle="->", color="tab:blue", lw=2))
    ax1.annotate("", xy=(1.3, 0), xytext=(0, 0), arrowprops=dict(arrowstyle="->", color="k", lw=2.5))
    ax1.text(1.3, -0.15, f"v = {speed_kmh:g} km/h", ha="center", va="top")
    arc = np.linspace(0, th, 50)
    ax1.plot(0.5 * np.cos(arc), 0.5 * np.sin(arc), color="tab:red")
    ax1.text(0.62 * np.cos(th / 2), 0.62 * np.sin(th / 2), r"$\theta$", color="tab:red", fontsize=13,
             ha="center", va="center")
    ax1.plot(0, 0, "ks", ms=10)
    ax1.set_xlim(-2.3, 2.3); ax1.set_ylim(-2.3, 2.3); ax1.set_aspect("equal"); ax1.axis("off")
    ax1.set_title("plane wave arriving at angle θ")
    # f_D vs theta
    tt = np.linspace(0, 360, 361)
    ax2.plot(tt, ss.doppler_shift_hz(fm, np.deg2rad(tt)), color="tab:blue", lw=2)
    ax2.plot(theta_deg, fd, "o", color="tab:red", ms=9)
    ax2.axhline(0, color="gray", lw=0.8)
    ax2.set_xlabel(r"angle of arrival $\theta$ (deg)"); ax2.set_ylabel(r"Doppler shift $f_D$ (Hz)")
    ax2.set_xticks(range(0, 361, 45)); ax2.set_xlim(0, 360); ax2.grid(alpha=0.3)
    ax2.set_title(rf"$f_m$ = {fm:.1f} Hz,  $f_D = f_m\cos\theta$ = {fd:+.1f} Hz")
    plt.tight_layout(); plt.show()

interact(show_doppler,
         speed_kmh=FloatSlider(value=60, min=1, max=300, step=1, description="v (km/h)"),
         fc_ghz=FloatSlider(value=2.0, min=0.7, max=28, step=0.1, description="f_c (GHz)"),
         theta_deg=FloatSlider(value=45, min=0, max=360, step=5, description="θ (deg)"));

## 2. Fading signal model: flat vs. frequency-selective
With $N$ plane waves of amplitude $C_n$, delay $\tau_n$ and Doppler $f_{D,n}$, the channel impulse response and its
Fourier transform over the delay variable $\tau$ are

$$h(t,\tau) = \sum_{n=1}^{N} C_n e^{-j\phi_n(t)}\,\delta(\tau-\tau_n), \qquad
\phi_n(t) = 2\pi\left[(f_c+f_{D,n})\tau_n - f_{D,n}t\right],$$
$$T(t,f) = \mathcal{F}_\tau\left[h(t,\tau)\right] = \sum_{n=1}^{N} C_n e^{-j\phi_n(t)} e^{-j2\pi f\tau_n}.$$

If the delay differences are small compared with the symbol duration, all $\tau_n\approx\hat\tau$ and
$|T(t,f)| = |h_{\mathrm{FLAT}}(t)|$: the gain fades in time (because the phases $\phi_n(t)$ still differ) but is the same
at every frequency — **flat fading**. If the delays are spread out, $|T(t,f)|$ also varies across the signal band —
**frequency-selective fading**. Below: the same 20 waves ($f_m = 50$ Hz) with delays spread over 50 ns vs. 5 µs,
shown over a 10 MHz band.

In [ ]:
rng = np.random.default_rng(7)
N, fm = 20, 50.0
theta = rng.uniform(0, 2 * np.pi, N)
C = np.ones(N) / np.sqrt(N)                    # equal amplitudes, unit total power
phase0 = rng.uniform(0, 2 * np.pi, N)          # 2 pi (f_c + f_D) tau_n  (mod 2 pi) is effectively random
t = np.linspace(0, 0.1, 200)                   # 100 ms
f = np.linspace(-5e6, 5e6, 301)                # +/- 5 MHz around f_c
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), sharey=True)
for ax, spread in zip(axes, [50e-9, 5e-6]):
    tau = rng.uniform(0, spread, N)
    T = ss.time_frequency_response(C, tau, ss.doppler_shift_hz(fm, theta), phase0, t, f)
    im = ax.pcolormesh(f / 1e6, t * 1e3, db20(T), vmin=-30, vmax=10, cmap="viridis", shading="auto")
    kind = "flat" if spread < 1e-6 else "frequency-selective"
    ax.set_title(f"delays spread over {spread * 1e6:g} µs: {kind}")
    ax.set_xlabel("frequency offset f (MHz)")
axes[0].set_ylabel("time t (ms)")
fig.colorbar(im, ax=axes, label=r"$|T(t,f)|$ (dB)")
plt.show()

Now look at a single snapshot $t = 0$. The left panel is the channel impulse response (a tapped delay line with
exponentially decaying average power), the right panel is $|T(0,f)|$ with a signal of bandwidth $B$ shaded.
Increase the maximum excess delay or the bandwidth and watch the in-band gain variation grow. (Week 6 makes
this precise with the rms delay spread $\sigma_\tau$ and the coherence bandwidth $B_c \approx 1/(5\sigma_\tau)$.)

In [ ]:
def show_selectivity(max_delay_us=1.0, bandwidth_mhz=5.0):
    rng = np.random.default_rng(3)
    n = 12
    tau = np.sort(np.r_[0, rng.uniform(0, 1, n - 1)]) * max_delay_us * 1e-6
    p = np.exp(-tau / (max_delay_us * 1e-6 / 3))                    # exponential power delay profile
    g = np.sqrt(p / p.sum()) * fading.iid_rayleigh(n, rng)          # Rayleigh path gains
    f = np.linspace(-10e6, 10e6, 2001)
    H = ss.time_frequency_response(g, tau, np.zeros(n), np.zeros(n), 0.0, f)[0]
    sig_tau = ss.rms_delay_spread(tau, p)
    inband = np.abs(f) <= bandwidth_mhz * 1e6 / 2
    var_db = np.ptp(db20(H[inband]))
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4), gridspec_kw={"width_ratios": [1, 1.6]})
    ax1.stem(tau * 1e6, np.abs(g) ** 2, basefmt=" ")
    ax1.set_xlabel(r"delay $\tau$ (µs)"); ax1.set_ylabel(r"path power $|C_n|^2$")
    ax1.set_title(rf"impulse response, $\sigma_\tau$ = {sig_tau * 1e6:.2f} µs"); ax1.grid(alpha=0.3)
    ax2.plot(f / 1e6, db20(H), color="tab:blue", lw=1.5)
    ax2.axvspan(-bandwidth_mhz / 2, bandwidth_mhz / 2, color="tab:orange", alpha=0.2, label=f"signal band B = {bandwidth_mhz:g} MHz")
    ax2.set_ylim(-35, 12); ax2.set_xlim(-10, 10)
    ax2.set_xlabel("frequency offset f (MHz)"); ax2.set_ylabel(r"$|T(0,f)|$ (dB)")
    ax2.set_title(f"gain varies by {var_db:.1f} dB across the signal band"); ax2.grid(alpha=0.3)
    ax2.legend(loc="lower right")
    plt.tight_layout(); plt.show()

interact(show_selectivity,
         max_delay_us=FloatSlider(value=1.0, min=0.02, max=10, step=0.02, description="max delay (µs)"),
         bandwidth_mhz=FloatSlider(value=5.0, min=0.2, max=20, step=0.2, description="B (MHz)"));

## 3. I/Q decomposition and the central limit theorem
For an unmodulated carrier the received signal splits into in-phase and quadrature parts,

$$h_I(t) = \sum_{n=1}^{N} C_n\cos\phi_n(t), \qquad h_Q(t) = \sum_{n=1}^{N} C_n\sin\phi_n(t), \qquad
r(t) = \sqrt{h_I^2(t)+h_Q^2(t)}.$$

With many paths and independent phases, the central limit theorem makes $h_I$ and $h_Q$ Gaussian, and the envelope
$r$ Rayleigh. Below each path has amplitude $C_n = 1/\sqrt{N}$ (so $\Omega_p = \mathrm{E}[r^2] = 1$) and a uniform
random phase. Left: one realisation of the phasor sum (head to tail); middle: histogram of $h_I$ against
$\mathcal{N}(0, \sigma^2 = 1/2)$; right: histogram of $r$ against the Rayleigh pdf. Try $N = 1, 2, 3, 6, 30$.

In [ ]:
def show_clt(n_paths=6):
    rng = np.random.default_rng(11)
    phi = rng.uniform(0, 2 * np.pi, n_paths)
    steps = np.exp(-1j * phi) / np.sqrt(n_paths)
    pts = np.r_[0, np.cumsum(steps)]
    a = ss.random_phasor_sum(n_paths, 20000, rng=1)
    fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
    ax = axes[0]
    for k in range(n_paths):
        ax.annotate("", xy=(pts[k + 1].real, pts[k + 1].imag), xytext=(pts[k].real, pts[k].imag),
                    arrowprops=dict(arrowstyle="->", color="tab:blue", lw=1.2))
    ax.annotate("", xy=(pts[-1].real, pts[-1].imag), xytext=(0, 0),
                arrowprops=dict(arrowstyle="->", color="tab:red", lw=2.5))
    lim = max(1.3, 1.1 * np.abs(pts).max())
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim); ax.set_aspect("equal"); ax.grid(alpha=0.3)
    ax.set_xlabel(r"$h_I$"); ax.set_ylabel(r"$h_Q$")
    ax.set_title(f"N = {n_paths} phasors, resultant r = {abs(pts[-1]):.2f}")
    x = np.linspace(-2.5, 2.5, 300)
    axes[1].hist(a.real, bins=60, range=(-2.5, 2.5), density=True, color="tab:blue", alpha=0.5, label="simulation")
    axes[1].plot(x, np.exp(-x**2) / np.sqrt(np.pi), "k", lw=2, label=r"$\mathcal{N}(0, 1/2)$")
    axes[1].set_xlabel(r"in-phase component $h_I$"); axes[1].set_ylabel("pdf"); axes[1].legend(loc="upper right")
    axes[1].set_title("in-phase component")
    r = np.linspace(0, 3, 300)
    axes[2].hist(np.abs(a), bins=60, range=(0, 3), density=True, color="tab:blue", alpha=0.5, label="simulation")
    axes[2].plot(r, ss.rayleigh_pdf(r), "k", lw=2, label=r"Rayleigh, $\Omega_p = 1$")
    axes[2].set_xlabel("envelope r"); axes[2].set_ylabel("pdf"); axes[2].legend(loc="upper right")
    axes[2].set_title("envelope")
    for ax in axes[1:]:
        ax.set_ylim(0, 1.6); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

interact(show_clt, n_paths=IntSlider(value=6, min=1, max=50, description="N paths"));

With $N = 1$ the envelope is constant, with $N = 2$ it piles up near $r = 0$ and $r = \sqrt 2$; already by
$N \approx 6$ the histogram is hard to distinguish from Rayleigh.

## 4. Rayleigh fading: envelope and power
With $h_I, h_Q \sim \mathcal{N}(0,\sigma^2)$ independent, the envelope is Rayleigh and its square — proportional to the
instantaneous received power — is exponential:

$$\mathrm{pdf}(r) = \frac{r}{\sigma^2}e^{-r^2/2\sigma^2} = \frac{2r}{\Omega_p}e^{-r^2/\Omega_p}, \qquad
\Omega_p = 2\sigma^2 = r_{\mathrm{rms}}^2, \qquad
\mathrm{pdf}_{r^2}(x) = \frac{1}{\Omega_p}e^{-x/\Omega_p},$$
$$\Pr(r<r_{\min}) = 1-\exp\left(-\frac{r_{\min}^2}{r_{\mathrm{rms}}^2}\right).$$

Here the samples come from the Clarke/Gans simulator (Section 8) with $\Omega_p = 1$.

In [ ]:
fs, fm = 1000.0, 50.0                               # 20 samples per Doppler period
h = np.concatenate([fading.clarke_gans(fm, 2**15, fs, rng=s) for s in range(4)])   # 131 s of fading
r = np.abs(h)
print(f"{len(h)} samples: E[r^2] = {np.mean(r**2):.3f} (Omega_p = 1),  E[r] = {r.mean():.3f} "
      f"(theory sqrt(pi)/2 = {np.sqrt(np.pi) / 2:.3f})")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
x = np.linspace(0, 3, 300)
axes[0].hist(r, bins=60, range=(0, 3), density=True, color="tab:blue", alpha=0.5, label="simulation")
axes[0].plot(x, ss.rayleigh_pdf(x), "k", lw=2, label="Rayleigh pdf")
axes[0].set_xlabel("envelope r"); axes[0].set_ylabel("pdf"); axes[0].set_title("envelope r = |h| is Rayleigh")
x = np.linspace(0, 6, 300)
axes[1].hist(r**2, bins=60, range=(0, 6), density=True, color="tab:blue", alpha=0.5, label="simulation")
axes[1].semilogy(x, ss.exponential_power_pdf(x), "k", lw=2, label="exponential pdf")
axes[1].set_xlabel(r"power $r^2$"); axes[1].set_ylabel("pdf (log scale)"); axes[1].set_title(r"power $r^2$ is exponential")
for ax in axes:
    ax.grid(alpha=0.3); ax.legend()
plt.tight_layout(); plt.show()

## 5. Fade margin and outage
The fade margin is how far the mean power sits above the minimum acceptable level:
$M = r_{\mathrm{rms}}^2 / r_{\min}^2$, $M_{\mathrm{dB}} = 10\log_{10}(r_{\mathrm{rms}}^2/r_{\min}^2)$. For Rayleigh fading
$\Pr(r<r_{\min}) = 1-e^{-1/M}$. The lecture's example asks for 1% outage:

$$\frac{r_{\min}^2}{r_{\mathrm{rms}}^2} = -\ln(0.99) \approx 0.01 \;\Longrightarrow\; M_{\mathrm{dB}} \approx 20\ \mathrm{dB}.$$

For small outage $1-e^{-1/M}\approx 1/M$, so every extra decade of reliability costs another 10 dB.

In [ ]:
print(f"{'outage':>8} {'margin (exact)':>15} {'1/P approx':>11} {'simulated outage':>17}")
for p in [0.1, 0.01, 1e-3, 1e-4]:
    m = ss.rayleigh_fade_margin_db(p)
    sim = np.mean(db20(r) < -m)                       # r_rms = 1, so r_min in dB is -M_dB
    print(f"{p:8.0e} {m:12.2f} dB {10 * np.log10(1 / p):8.1f} dB {sim:17.2e}")

m_db = np.linspace(-5, 40, 200)
fig, ax = plt.subplots(figsize=(7.5, 4.3))
ax.semilogy(m_db, ss.rayleigh_outage_probability(m_db), "k", lw=2, label=r"theory $1-e^{-1/M}$")
lvl = np.arange(-5, 36, 2.5)
ax.semilogy(lvl, [np.mean(db20(r) < -m) for m in lvl], "o", color="tab:blue", ms=7, label="simulation (Section 4 samples)")
m1 = ss.rayleigh_fade_margin_db(0.01)
ax.plot([m_db[0], m1, m1], [0.01, 0.01, 1e-5], "--", color="tab:red", lw=1.5)
ax.annotate(f"1% outage needs {m1:.2f} dB", (m1, 0.01), xytext=(m1 + 2, 0.05), color="tab:red",
            arrowprops=dict(arrowstyle="->", color="tab:red"))
ax.set_ylim(1e-5, 1); ax.set_xlim(-5, 40)
ax.set_xlabel(r"fade margin $M_{\mathrm{dB}}$ (dB)"); ax.set_ylabel(r"outage $\Pr(r < r_{\min})$")
ax.set_title("Rayleigh fading: outage vs. fade margin"); ax.grid(True, which="both", alpha=0.3); ax.legend(loc="upper right")
plt.tight_layout(); plt.show()

**Interference example.** If the desired signal *and* the interferer are both Rayleigh-faded, the instantaneous SIR
$\Lambda$ is a ratio of two exponentials, and $\Pr(\Lambda < \Lambda_{\min}) = \Lambda_{\min}/(\Lambda_{\min}+\bar\Lambda)$.
With a mean SIR $\bar\Lambda = 10$ dB and threshold 0 dB this is $1/11 \approx 0.09$ — much worse than the 1%
you would get if only the desired signal faded with the same 10 dB margin.

In [ ]:
rng = np.random.default_rng(0)
n = 200_000
S = 10 * rng.exponential(size=n)     # desired power, mean 10 (10 dB above the interferer)
I = rng.exponential(size=n)          # interferer power, mean 1
print(f"Pr(SIR < 0 dB): theory {ss.sir_outage_rayleigh(0, 10):.4f}, simulation {np.mean(S / I < 1):.4f}")
print(f"only the desired signal fades (10 dB margin): {ss.rayleigh_outage_probability(10):.4f}")

## 6. Ricean fading and the $K$-factor
With a line-of-sight component of amplitude $A$ aligned with the real axis, $\mathrm{Re}(h)\sim\mathcal{N}(A,\sigma^2)$ and
$\mathrm{Im}(h)\sim\mathcal{N}(0,\sigma^2)$, and the envelope is Ricean with

$$K = \frac{\text{power in LOS component}}{\text{power in random components}} = \frac{A^2}{2\sigma^2}, \qquad
\mathrm{pdf}(r) = \frac{2(K+1)r}{\Omega_p}\exp\left(-K-\frac{(K+1)r^2}{\Omega_p}\right)
I_0\left(2r\sqrt{\frac{K(K+1)}{\Omega_p}}\right),$$

where $\Omega_p = A^2 + 2\sigma^2$ (here 1). $K = 0$ is Rayleigh; $K\to\infty$ is no fading at all. The samples come from
`fading.rician_sos` with the LOS wave arriving broadside ($\theta = 90^\circ$, no Doppler), so it stays on the real axis.
The right panel is the outage/fade-margin curve of Section 5 for this $K$.

In [ ]:
def show_ricean(k_factor=3.0):
    t = np.arange(2000) / 1000.0                    # 2 s at 1 kHz, f_m = 100 Hz
    h = np.concatenate([fading.rician_sos(100, t, k_factor, los_angle=np.pi / 2, rng=s) for s in range(15)])
    r = np.abs(h)
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.3))
    ax = axes[0]
    ax.plot(h.real[::10], h.imag[::10], ".", ms=2, alpha=0.4, color="tab:blue")
    A = np.sqrt(k_factor / (k_factor + 1))
    ax.plot(A, 0, "*", color="tab:red", ms=15, label=f"LOS, A = {A:.2f}")
    ax.set_xlim(-2.2, 2.2); ax.set_ylim(-2.2, 2.2); ax.set_aspect("equal"); ax.grid(alpha=0.3)
    ax.set_xlabel(r"Re$(h)$"); ax.set_ylabel(r"Im$(h)$"); ax.set_title("complex gain samples"); ax.legend(loc="upper left")
    x = np.linspace(0, 2.5, 300)
    axes[1].hist(r, bins=60, range=(0, 2.5), density=True, color="tab:blue", alpha=0.5, label="simulation")
    axes[1].plot(x, ss.rician_pdf(x, k_factor), "k", lw=2, label=f"Ricean, K = {k_factor:g}")
    axes[1].plot(x, ss.rayleigh_pdf(x), "--", color="gray", lw=1.5, label="Rayleigh (K = 0)")
    axes[1].set_xlabel("envelope r"); axes[1].set_ylabel("pdf"); axes[1].set_ylim(0, 3.2)
    axes[1].set_title("envelope pdf"); axes[1].legend(loc="upper right"); axes[1].grid(alpha=0.3)
    m_db = np.linspace(-5, 30, 200)
    ax = axes[2]
    ax.semilogy(m_db, ss.rician_outage_probability(m_db, k_factor), "k", lw=2, label=f"Ricean, K = {k_factor:g}")
    ax.semilogy(m_db, ss.rayleigh_outage_probability(m_db), "--", color="gray", lw=1.5, label="Rayleigh")
    lvl = np.arange(-5, 26, 2.5)
    sim = np.array([np.mean(db20(r) < -m) for m in lvl])
    ax.semilogy(lvl[sim > 0], sim[sim > 0], "o", color="tab:blue", ms=6, label="simulation")
    m1 = ss.rician_fade_margin_db(0.01, k_factor)
    ax.axhline(0.01, color="tab:red", ls=":", lw=1)
    ax.axvline(m1, color="tab:red", ls=":", lw=1)
    ax.set_ylim(1e-4, 1); ax.set_xlim(-5, 30); ax.grid(True, which="both", alpha=0.3)
    ax.set_xlabel(r"fade margin $M_{\mathrm{dB}}$ (dB)"); ax.set_ylabel(r"$\Pr(r < r_{\min})$")
    ax.set_title(f"1% outage: {m1:.1f} dB margin (Rayleigh: 20.0 dB)"); ax.legend(loc="upper right")
    plt.tight_layout(); plt.show()

interact(show_ricean, k_factor=FloatSlider(value=3, min=0, max=30, step=0.5, description="K"));

## 7. Doppler power spectrum and autocorrelation
Each plane wave at angle $\alpha$ lands at frequency $f = f_c + f_m\cos\alpha$. Equating the power received over
$d\alpha$ with the power in $df$, with uniform angle of arrival $p(\alpha) = 1/2\pi$ and a dipole gain $G(\alpha) = 1.5$, gives
the U-shaped (Clarke/Jakes) spectrum

$$S(f) = \frac{1.5}{\pi f_m\sqrt{1-\left(\frac{f-f_c}{f_m}\right)^2}}, \qquad |f-f_c| < f_m .$$

The simulators here use unit gain and $\Omega_p = 1$ at baseband, i.e. $S(f) = 1/\bigl(\pi f_m\sqrt{1-(f/f_m)^2}\bigr)$
(which integrates to 1). Below: Welch PSD estimates of both simulators against this curve. The theoretical spectrum is
infinite at $\pm f_m$; the estimate is finite because of its frequency resolution. Clarke/Gans matches closely.
A sum-of-sinusoids realisation with fixed angles $\theta_n$ has a *line* spectrum (64 tones at $f_m\cos\theta_n$), so
even averaged over 8 realisations its estimate is jagged.

In [ ]:
fs, fm, n = 1000.0, 100.0, 2**14
t = np.arange(n) / fs
sims = {
    "Clarke/Gans": [fading.clarke_gans(fm, n, fs, rng=s) for s in range(8)],
    "sum of sinusoids (64 paths)": [fading.rayleigh_sos(fm, t, n_paths=64, rng=s) for s in range(8)],
}
fig, ax = plt.subplots(figsize=(8, 4.5))
x = np.linspace(-0.999, 0.999, 1000)
ax.semilogy(x, ss.doppler_psd(x * fm, fm) * fm, color="k", lw=3, alpha=0.6, label=r"theory $1/(\pi f_m\sqrt{1-(f/f_m)^2})$")
for (name, hs), c in zip(sims.items(), ["tab:blue", "tab:orange"]):
    psd = np.mean([welch(h, fs=fs, nperseg=1024, return_onesided=False)[1] for h in hs], axis=0)
    fw = welch(hs[0], fs=fs, nperseg=1024, return_onesided=False)[0]
    k = np.argsort(fw)
    ax.semilogy(fw[k] / fm, psd[k] * fm, color=c, lw=1.5, label=name + " (Welch)")
ax.set_xlim(-1.5, 1.5); ax.set_ylim(1e-4, 30)
ax.set_xlabel(r"normalised Doppler frequency $f/f_m$"); ax.set_ylabel(r"$f_m\,S(f)$ (normalised PSD)")
ax.set_title(rf"Doppler power spectrum, $f_m$ = {fm:g} Hz"); ax.grid(True, which="both", alpha=0.3); ax.legend(loc="lower center")
plt.tight_layout(); plt.show()

The Doppler spectrum and the time autocorrelation of $h(t)$ are a Fourier pair. For the U-shaped spectrum

$$R(\tau) = \mathrm{E}[h(t)h^*(t+\tau)] = \Omega_p\,J_0(2\pi f_m\tau),$$

so the fading decorrelates after roughly $\tau \approx 0.4/f_m$ (the first zero of $J_0$ is at $f_m\tau = 0.383$) — a
preview of the coherence time $T_c$ in Week 6. The sum-of-sinusoids model with few paths only approximates $J_0$.

In [ ]:
max_lag = int(3 * fs / fm)
lags = np.arange(max_lag + 1) / fs
few = [fading.rayleigh_sos(fm, t, n_paths=8, rng=s) for s in range(8)]
fig, ax = plt.subplots(figsize=(8, 4.2))
ax.plot(lags * fm, ss.clarke_autocorrelation(lags, fm), "k", lw=2.5, label=r"theory $J_0(2\pi f_m\tau)$")
for (name, hs), c in zip(list(sims.items()) + [("sum of sinusoids (8 paths)", few)], ["tab:blue", "tab:orange", "tab:green"]):
    R = np.mean([ss.empirical_autocorrelation(h, max_lag) for h in hs], axis=0)
    ax.plot(lags * fm, R.real, "o-", ms=4, lw=1, color=c, label=name)
ax.axvline(2.405 / (2 * np.pi), color="gray", ls=":")
ax.text(0.40, 0.85, r"first zero $f_m\tau = 0.383$", color="gray")
ax.set_xlabel(r"normalised lag $f_m\tau$"); ax.set_ylabel(r"$\mathrm{Re}\,R(\tau)/\Omega_p$")
ax.set_title(rf"autocorrelation of the fading gain, $f_m$ = {fm:g} Hz"); ax.grid(alpha=0.3); ax.legend(loc="upper right")
plt.tight_layout(); plt.show()

## 8. Fading simulators: sum of sinusoids vs. Clarke/Gans
The homework builds a Rayleigh fading simulator. Two approaches are available in `ece512.fading`:

* **Sum of sinusoids** (`rayleigh_sos`): Clarke's model directly — $h(t) = \frac{1}{\sqrt N}\sum_n e^{j(2\pi f_m\cos\theta_n t + \phi_n)}$.
* **Clarke/Gans** (`clarke_gans`, Rappaport Fig. 5.24): complex Gaussian noise shaped in frequency by $\sqrt{S(f)}$ and
  transformed to time.

Below: the envelope $20\log_{10}(r/r_{\mathrm{rms}})$ for $f_m = 20$ Hz (e.g. 5.2 GHz at ~4 km/h) and $f_m = 200$ Hz
(e.g. 2 GHz at ~108 km/h). Deep fades of 20–30 dB occur roughly every half wavelength of travel, i.e. about every
$1/(2f_m)$ seconds, and the faster channel fades ten times as often.

In [ ]:
fs, dur = 5000.0, 0.5
t = np.arange(int(fs * dur)) / fs
fig, axes = plt.subplots(2, 2, figsize=(13, 6.5), sharex=True, sharey=True)
for row, fm in zip(axes, [20.0, 200.0]):
    h_sos = fading.rayleigh_sos(fm, t, n_paths=64, rng=4)
    h_cg = fading.clarke_gans(fm, 2**15, fs, rng=4)[: len(t)]
    h_cg = h_cg / np.sqrt(np.mean(np.abs(h_cg) ** 2))     # normalise the 0.5 s excerpt to r_rms = 1
    for ax, h, name, c in zip(row, [h_sos, h_cg], ["sum of sinusoids (64 paths)", "Clarke/Gans"], ["tab:blue", "tab:orange"]):
        ax.plot(t * 1e3, db20(h), color=c, lw=1.1)
        ax.axhline(-20, color="tab:red", ls=":", lw=1.2)
        frac = np.mean(db20(h) < -20)
        ax.set_title(rf"{name}, $f_m$ = {fm:g} Hz  ({100 * frac:.1f}% of time below −20 dB)", fontsize=11)
        ax.set_ylim(-40, 10); ax.grid(alpha=0.3)
for ax in axes[:, 0]:
    ax.set_ylabel(r"$20\log_{10}(r/r_{\mathrm{rms}})$ (dB)")
for ax in axes[1]:
    ax.set_xlabel("time t (ms)")
plt.tight_layout(); plt.show()

## Try it yourself
* A 28 GHz mmWave user in a car at 100 km/h: what is $f_m$, and how often (in ms) do you expect a deep fade? How does
  that compare with a 1 ms scheduling slot?
* In Section 3, how many paths $N$ are needed before the simulated $\Pr(r < 0.1)$ is within 10% of the Rayleigh value
  $1-e^{-0.01}$? Why do the deep fades converge more slowly than the body of the distribution?
* Using `ss.rician_fade_margin_db`, find the fade margin for 0.1% outage at $K = 0, 3, 10$ (linear, not dB).
  How many dB does a strong LOS component save compared with Rayleigh, and why does the saving grow as the
  outage target gets stricter?
* Reduce `n_paths` in the sum-of-sinusoids simulator (Sections 7–8) to 4 or 8. Which statistics — envelope pdf,
  Doppler spectrum, autocorrelation — degrade first, and why is the Clarke/Gans method immune to this?